# 胸鳍对状态：距离和两个朝向角

重新计算物理量从这里开始。每一对鱼三个数，三对一共九列。定义照
O'Shaughnessy et al., *Dynamics of Dominance in Interacting Zebrafish*, PRX Life **2**, 043006 (2024),
https://doi.org/10.1103/PRXLife.2.043006 ，Fig. 1(c)、Fig. 2(c) 和 Methods 的身体点一节。

他们的鱼是两条雄性斑马鱼。这里把同一套坐标用到一雌两雄的三对上：雄1–雌、雄2–雌、雄1–雄2。
K 组三天是同一三条鱼，按实验分开算，不合并。

## 从论文拿来的算法

三条身体点和现有骨架一致：嘴、胸鳍、尾。胸鳍点是位置（两胸鳍基部的中点）。朝向是胸鳍指向嘴的向量。

一对鱼只留三个数：

1. **D_PP**：两鱼胸鳍点的三维欧氏距离，毫米。竖直分开算在这个距离里。
2. **θ_a**：鱼 a 的胸鳍→嘴向量在水平面 XY 上的投影，转到「自己的胸鳍指向对方胸鳍」这条水平连线的有符号角。
3. **θ_b**：鱼 b 的同一个角。

逆时针为正（+Z 向上）。0° 是吻端在水平面上指向对方胸鳍。+90° 是对方在这条鱼的左侧。180° 是吻端背对对方。范围是 (−180°, 180°]。
水平朝向或水平连线短于 1 mm 时，这个角留空：鱼几乎竖直，或两鱼胸鳍在水平面上叠在一起，方向是噪声。距离仍然保留。

绝对位置和两鱼一起转过的水平角不会改变这三个数。论文用主成分把 18 维关键点收成这三项，并看到距离变化慢、朝向变化快。这次直接用他们给出的三个可解释坐标，不再做那一步主成分。

## 这次不搬的部分

胜负标签、1 分钟滑窗聚类、Jensen–Shannon 距离、Infomap 机动，都是斑马鱼双雄格斗的后续分析。配偶选择先把九列算出来，再决定要不要用同样的分布去看互动。
他们的 5 帧线性插值和 9 帧 Savitzky–Golay 也不用。一帧能不能算，沿用身份审阅：分析起点之前、没看过、不可用，或者三条鱼没有落在三条不同轨迹上，整帧留空。缺嘴只让对应的角为空，胸鳍还在时距离照算。

## 九列

| 对 | 胸鳍距离 | 第一条鱼的 θ | 第二条鱼的 θ |
|---|---|---|---|
| 雄1–雌 | `d_pp_M1_F` | `theta_M1_on_F` | `theta_F_on_M1` |
| 雄2–雌 | `d_pp_M2_F` | `theta_M2_on_F` | `theta_F_on_M2` |
| 雄1–雄2 | `d_pp_M1_M2` | `theta_M1_on_M2` | `theta_M2_on_M1` |

同一条鱼对着不同对象，θ 不是同一个数。读的时候：两角都接近 0 是面对面；一个接近 0、另一个接近 ±180° 是一鱼朝向对方、对方背对；两角一正一负且绝对值接近，是朝向大致平行；两角同号且绝对值接近，是朝向大致反平行。论文里追击、并排、近距离绕圈就是用这种读法从 (D_PP, θ_a, θ_b) 里认出来的。

已有的 `head_A_to_B` 用质心连线，在三维里取 0–180° 的无符号角，左右叠在一起。上面九列用胸鳍点，角只在水平面上，并保留左右。

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "scripts" / "common" / "local_paths.py").exists():
            return d
    raise FileNotFoundError(f"找不到仓库根目录（从 {start} 向上没有 pyproject.toml）")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

PAIRS = (("M1", "F"), ("M2", "F"), ("M1", "M2"))
NAMES = ("F", "M1", "M2")
MIN_XY_MM = 1.0


def signed_planar_angle_deg(heading, line, min_xy_mm=MIN_XY_MM):
    """Signed angle from ``heading`` to ``line`` after dropping Z.

    Counterclockwise is positive. 0 means the heading points along ``line``.
    A horizontal piece shorter than ``min_xy_mm`` is missing.
    """
    h = np.asarray(heading, float)[..., :2]
    r = np.asarray(line, float)[..., :2]
    nh = np.linalg.norm(h, axis=-1)
    nr = np.linalg.norm(r, axis=-1)
    cross = h[..., 0] * r[..., 1] - h[..., 1] * r[..., 0]
    dot = np.sum(h * r, axis=-1)
    ang = np.degrees(np.arctan2(cross, dot))
    out = np.full(np.shape(nh), np.nan)
    good = (nh >= min_xy_mm) & (nr >= min_xy_mm) & np.isfinite(ang)
    out[good] = ang[good]
    return out


def pectoral_pair_table(nodes, frames, mouth=0, pectoral=1, min_xy_mm=MIN_XY_MM):
    """Nine pair coordinates. ``nodes`` is ``(frame, 3, node, 3)`` in order F, M1, M2, millimetres."""
    slot = {name: i for i, name in enumerate(NAMES)}
    pec = nodes[:, :, pectoral, :]
    heading = nodes[:, :, mouth, :] - pec
    cols = {"frame": np.asarray(frames)}
    for a, b in PAIRS:
        ia, ib = slot[a], slot[b]
        delta = pec[:, ib] - pec[:, ia]
        dist = np.linalg.norm(delta, axis=-1)
        finite = np.isfinite(pec[:, ia]).all(1) & np.isfinite(pec[:, ib]).all(1)
        cols[f"d_pp_{a}_{b}"] = np.where(finite, dist, np.nan)
        cols[f"theta_{a}_on_{b}"] = signed_planar_angle_deg(heading[:, ia], delta, min_xy_mm)
        cols[f"theta_{b}_on_{a}"] = signed_planar_angle_deg(heading[:, ib], -delta, min_xy_mm)
    return pd.DataFrame(cols)


def _pose(pec, mouth):
    nodes = np.full((1, 3, 3, 3), np.nan)
    for fish, origin, tip in zip(range(3), pec, mouth):
        if origin is None:
            continue
        nodes[0, fish, 1] = origin
        nodes[0, fish, 0] = tip
    return nodes


facing = pectoral_pair_table(
    _pose([(0, 0, 0), (40, 0, 0), None], [(12, 0, 0), (52, 0, 0), None]), [0])
assert facing.loc[0, "d_pp_M1_F"] == 40
assert facing.loc[0, "theta_F_on_M1"] == 0
assert facing.loc[0, "theta_M1_on_F"] == 180

left = pectoral_pair_table(
    _pose([(0, 0, 0), (0, 40, 0), None], [(12, 0, 0), (0, 52, 0), None]), [0])
assert left.loc[0, "theta_F_on_M1"] == 90

vertical = pectoral_pair_table(
    _pose([(0, 0, 0), (40, 0, 0), None], [(0, 0, 12), (40, 0, 12), None]), [0])
assert vertical.loc[0, "d_pp_M1_F"] == 40
assert np.isnan(vertical.loc[0, "theta_F_on_M1"])

print("角度约定：0° 朝向对方，+90° 对方在左侧，180° 背对。竖直朝向时距离保留、角为空。")
print("仓库", REPO)

## 读一整个实验

坐标来自身份审阅包里的 `nodes3d_filled`（没有则用 `nodes3d`），单位毫米。
谁是雌鱼、雄1、雄2，用审阅账本里的静态标签；轨迹对调用账本里的 swap 换回来。
掩码和 `scripts.analysis.behavior_dataset.build_trial` 相同，所以空帧和已经导出的行为表对齐。

`EXPERIMENT` 换成别的文件夹名即可。下面先列出有身份账本的实验。

In [ ]:
import json

from scripts.analysis.behavior_dataset import find_package, resolve_roots
from scripts.analysis import behavior_features as F
from scripts.review import id_segments as S
from scripts.review.ledger import (
    COVERAGE_COLUMNS, EVENT_COLUMNS, analysis_start, owner_series,
    read_csv, static_labels, swap_list, watched_intervals,
)

_config, REVIEW, _out = resolve_roots(None)
EXPERIMENTS = sorted({
    p.parents[2].name
    for p in REVIEW.glob("*/*/id_pass/human_events.csv")
    if p.parents[3] == REVIEW
})
print("审阅根目录", REVIEW)
print("有身份账本的实验:")
for name in EXPERIMENTS:
    print(" ", name)

EXPERIMENT = "20251225_K_1_2_FCMC"


def load_named(experiment: str):
    package = find_package(REVIEW, experiment)
    led = package / "id_pass"
    with np.load(package / "review_data.npz") as z:
        frames = np.asarray(z["frames"], int)
        fish_ids = [str(f) for f in z["fish_ids"]]
        node_names = [str(n) for n in z["node_names"]]
        key = "nodes3d_filled" if "nodes3d_filled" in z.files else "nodes3d"
        nodes = np.asarray(z[key], float)
        xyz = np.asarray(z["xyz"], float)
        present = np.asarray(z["view_present"])
        interp = np.asarray(z["is_interpolated"])
    missing = [n for n in ("mouth", "pectoral", "tail") if n not in node_names]
    if missing:
        raise RuntimeError(f"{package} nodes are {node_names}; missing {missing}")
    manifest = json.loads((package.parent / "proxy" / "proxy_manifest.json").read_text(encoding="utf-8"))
    fps = float(manifest["views"]["top_cam"]["source_fps"])
    events = read_csv(led / "human_events.csv", EVENT_COLUMNS)
    coverage = read_csv(led / "coverage.csv", COVERAGE_COLUMNS)
    labels = static_labels(events, fish_ids)
    start = analysis_start(events)
    if start is None:
        raise RuntimeError(f"{package} has no analysis_start")
    swaps = swap_list(events, fish_ids)
    danger = S.danger_map(xyz, present, interp, fps)
    status = S.frame_status(
        int(frames[0]), len(frames), start, danger["danger"], danger["runs"],
        watched_intervals(coverage), [f for f, _, _ in swaps], S.manual_ranges(events), fps,
    )
    usable = np.isin(status, (S.WATCHED_SAFE, S.WATCHED_DANGER))
    owner = owner_series(len(fish_ids), swaps, frames)
    named, valid = F.named_nodes(nodes, owner, fish_ids, labels)
    named[~usable | ~valid] = np.nan
    info = {
        "experiment": experiment,
        "package": str(package),
        "node_source": key,
        "fps": fps,
        "analysis_start": int(start),
        "frames": int(len(frames)),
        "kept_frames": int(usable.sum()),
        "labels": labels,
    }
    return named, frames, node_names, info


named, frames, node_names, info = load_named(EXPERIMENT)
mouth = node_names.index("mouth")
pectoral = node_names.index("pectoral")
table = pectoral_pair_table(named, frames, mouth, pectoral)
print(json.dumps({k: v for k, v in info.items() if k != "labels"}, ensure_ascii=False, indent=2))
print("labels", info["labels"])

In [ ]:
def coverage(table: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(table)
    for a, b in PAIRS:
        dist = table[f"d_pp_{a}_{b}"].to_numpy()
        ta = table[f"theta_{a}_on_{b}"].to_numpy()
        tb = table[f"theta_{b}_on_{a}"].to_numpy()
        both = np.isfinite(dist) & np.isfinite(ta) & np.isfinite(tb)
        rows.append({
            "pair": f"{a}-{b}",
            "distance_frames": int(np.isfinite(dist).sum()),
            "both_angles_frames": int(both.sum()),
            "distance_without_angle": int((np.isfinite(dist) & ~(np.isfinite(ta) & np.isfinite(tb))).sum()),
            "distance_median_mm": float(np.nanmedian(dist)),
            "share_of_rows": round(float(both.sum()) / n, 4),
        })
    return pd.DataFrame(rows)


def heading_scale_mm(nodes, mouth, pectoral, tail):
    """Median finite segment lengths, as a scale for reading D_PP. Not one of the nine columns."""
    rows = []
    for name, i in zip(NAMES, range(3)):
        pec = nodes[:, i, pectoral]
        rows.append({
            "fish": name,
            "mouth_pectoral_mm": float(np.nanmedian(np.linalg.norm(nodes[:, i, mouth] - pec, axis=1))),
            "pectoral_tail_mm": float(np.nanmedian(np.linalg.norm(nodes[:, i, tail] - pec, axis=1))),
        })
    return pd.DataFrame(rows)


tail = node_names.index("tail")
display(coverage(table))
display(heading_scale_mm(named, mouth, pectoral, tail))
table.head(3)

## 看一小段

默认从分析起点起 20 秒，166 fps 下大约 3320 帧。空帧把线断开。
三行是三对。左列是胸鳍距离，右列是这一对的两个 θ。

In [ ]:
import matplotlib.pyplot as plt

START = int(info["analysis_start"])
SECONDS = 20.0


def plot_window(table, fps, start, seconds, experiment):
    end = start + seconds * fps
    part = table[(table["frame"] >= start) & (table["frame"] < end)].copy()
    t = (part["frame"].to_numpy() - start) / fps
    fig, axes = plt.subplots(3, 2, figsize=(11, 7), sharex=True, layout="constrained")
    for row, (a, b) in enumerate(PAIRS):
        axes[row, 0].plot(t, part[f"d_pp_{a}_{b}"], color="#222222", lw=0.8)
        axes[row, 0].set_ylabel(f"{a}-{b}\nD_PP (mm)")
        axes[row, 1].plot(t, part[f"theta_{a}_on_{b}"], color="#c51b8a", lw=0.8, label=f"theta {a}")
        axes[row, 1].plot(t, part[f"theta_{b}_on_{a}"], color="#2c7fb8", lw=0.8, label=f"theta {b}")
        axes[row, 1].axhline(0, color="0.7", lw=0.6)
        axes[row, 1].set_ylabel("theta (deg)")
        axes[row, 1].set_ylim(-180, 180)
        axes[row, 1].legend(loc="upper right", fontsize=8, frameon=False)
    axes[-1, 0].set_xlabel(f"seconds after frame {start}")
    axes[-1, 1].set_xlabel(f"seconds after frame {start}")
    fig.suptitle(f"{experiment}  pectoral distance and planar heading angles")
    return fig


plot_window(table, info["fps"], START, SECONDS, info["experiment"])